# Parquet

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Parquet is a columnar, typed file format for analytics, and the default storage format for Spark, Trino, Athena, and lakehouse tables.

## What is Apache Parquet?

Apache Parquet is an open-source columnar file format designed for analytical workloads. Values are stored by column inside row groups, each column chunk carries statistics (min, max, null count), and a footer holds the schema and metadata. Query engines read only the columns and row groups they need, which is why Parquet scans are cheaper than row-oriented files.

Under the hood a Parquet file is a small metadata tree wrapped around blocks of column data. The file is split into row groups, a row group holds one column chunk per column, and each column chunk is split into pages. The footer is written last and records the schema, the offset and size of every row group and column chunk, the encodings used, and min/max/null-count statistics. A reader opens the footer first, then seeks only to the byte ranges it needs.

Example: a query that selects `customer_id` and `amount` never touches a large `description` or `payload` column, and a filter on `order_date` can skip row groups whose min/max range cannot match.

Pages are the compression and encoding unit. A page is either a dictionary page or a data page. Dictionary encoding replaces repeated values with small integer keys; run-length encoding (RLE) and bit-packing then compress those keys and the definition and repetition levels that mark nulls and nesting. Integers and timestamps often use delta or delta-binary-packed encoding. When a dictionary grows past the dictionary page limit, the writer falls back to plain encoding for the rest of the chunk.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# Minimal shape of a Parquet dataset written by Spark
# s3://acme-curated/orders/ds=2026-10-04/part-00000-....snappy.parquet
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("parquet-demo").getOrCreate()

orders = spark.read.json("s3://acme-raw/orders/ds=2026-10-04/")

orders.write.mode("overwrite").parquet("s3://acme-curated/orders/ds=2026-10-04/")

Parquet is a file format, not a table format. It has no transactions, no row-level deletes, and no catalog. Iceberg, Delta Lake, and Hudi add those table semantics on top of Parquet files.

Concrete layout: a 1 GB table with 128 MB row groups becomes 8 row groups, and 30 columns produce 240 column chunks. A query that selects 2 columns and filters on one reads roughly 2/30 of the column bytes plus the footer, and it can skip any row group whose min/max for the filter column cannot match.

Common mistake: treating Parquet as a queryable index. Statistics are coarse (per column chunk, optionally per page) and only help when data is sorted or clustered on the filter column. An unsorted file has overlapping min/max ranges, so no row group can be skipped. Parquet gives you cheap columns, not free filtering.

## Why is Parquet better than CSV or JSON for analytics?

CSV and JSON are row-oriented text formats. They are human-readable but expensive: every query parses every row, every column, and often every byte. Parquet is binary, typed, columnar, compressed, and splittable, so engines like Spark, Hive, Athena, BigQuery external tables, Trino, and DuckDB scan far less data.

The cost model is bytes read, not rows read. A row-oriented file must be read end to end to answer any query because a row is stored contiguously. A columnar file stores each column separately, so an engine reads only the projected columns, decompresses only those chunks, and decodes only the values it needs. Typed columns also compress far better than text: integers and timestamps are bit-packed, repeated values are dictionary-encoded, and each column chooses its own codec.

A realistic comparison for the same 200 GB of order events:

| Aspect | JSON / CSV | Parquet |
|---|---|---|
| Storage size | 200 GB | 40-70 GB with Snappy, less with Zstd |
| Scan for 2 of 30 columns | reads every column | reads those 2 column chunks |
| Filter on one date | full scan, no skipping | skips row groups via min/max statistics |
| Schema | inferred per run | embedded in the footer |
| Splittable | often not (gzip CSV) | yes, splittable per row group |
| Nested fields | re-parsed as text | native struct/array/map types |

Worked example for 200 GB of JSON order events with 30 columns. Parquet with Snappy is roughly 45 GB (about 4.4x smaller); moving cold columns to Zstd drops it to about 35 GB. A query that scans 2 of 30 columns reads about 3 GB instead of 45 GB, so projection alone is a 15x cut before any skipping. Partition pruning on one day out of 365 cuts it again by another order of magnitude. The gains compound in this order: projection, partitions, row-group statistics, then compression.

The practical result: converting a JSON landing zone to partitioned Parquet commonly cuts both storage and warehouse scan cost, and removes a whole class of type-inference bugs.

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow reads only the columns you name, and only the row groups that can match
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# Column projection: the reader never touches the other 28 columns
table = pq.read_table(
    "orders/ds=2026-10-04/part-00000.snappy.parquet",
    columns=["customer_id", "amount"],
)

# Predicate pushdown through the dataset API: filters prune row groups by statistics
dataset = ds.dataset("orders/", format="parquet", partitioning="hive")
scanner = dataset.scanner(
    columns=["customer_id", "amount"],
    filter=(ds.field("order_date") == "2026-10-04") & (ds.field("country") == "DE"),
)
print(scanner.head(5))

Common mistake: converting to Parquet but keeping a schema that defeats it. A table where analysts always select all 30 columns, or one wide `payload` string that holds the real data, gains little. Columnar formats pay off when queries touch a subset of narrow, typed columns.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Compare the same data as JSON and as Parquet
events = spark.read.json("s3://acme-raw/events/2026/10/04/")

events.write.mode("overwrite").json("s3://acme-cmp/events_json/")
events.write.mode("overwrite").parquet("s3://acme-cmp/events_parquet/")

# Query only two columns and filter by date: Parquet reads a fraction of the bytes
parquet_scan = spark.read.parquet("s3://acme-cmp/events_parquet/")
parquet_scan.select("customer_id", "amount").where("event_date = '2026-10-04'").explain("cost")

## How do you write and read Parquet with PySpark?

Ask questions about `mode`, `partitionBy`, compression, and file count in interviews. The snippet below is a production-shaped write:

A PySpark write is a distributed sequence of per-task Parquet writers. Each task buffers rows and flushes a row group once it reaches the configured row-group size (128 MB by default); at the end of the task it writes the footer and closes the file. The number of output files equals the number of tasks that have data, which is why a shuffle or `repartition` controls file count. Compression is applied per page and dictionary encoding is enabled by default.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("orders-parquet").getOrCreate()

orders = (
    spark.read.json("s3://acme-raw/orders/")
    .withColumn("order_date", F.to_date("created_at"))
    .filter(F.col("order_date") == "2026-10-04")
)

# One logical partition per day, deterministic writes, compressed
(
    orders.repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .option("compression", "snappy")
    .parquet("s3://acme-curated/orders/")
)

# Read it back and let the engine prune partitions and row groups
curated = spark.read.parquet("s3://acme-curated/orders/")
(
    curated.filter(F.col("order_date") == "2026-10-04")
    .groupBy("country")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
    .show()
)

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: write with an explicit schema and explicit layout knobs
import pyarrow as pa
import pyarrow.parquet as pq

schema = pa.schema([
    ("order_id", pa.int64()),
    ("customer_id", pa.int64()),
    ("country", pa.dictionary(pa.int32(), pa.string())),
    ("amount", pa.decimal128(12, 2)),
    ("order_ts", pa.timestamp("us", tz="UTC")),
])

table = pa.Table.from_pylist(rows, schema=schema)

pq.write_table(
    table,
    "orders/ds=2026-10-04/part-00000.parquet",
    row_group_size=1_000_000,        # rows per row group, not bytes
    data_page_size=1 * 1024 * 1024,  # target page size before compression
    compression="zstd",
    compression_level=3,
    use_dictionary=True,
    write_statistics=True,
)

Mode rules that matter:

+ `overwrite` replaces the whole target (and with `partitionBy`, only the touched partitions under dynamic partition overwrite).
+ `append` is cheap but creates duplicates if a task reruns, which is why idempotency belongs to the table format or to a partition-replace pattern.
+ `repartition(col)` yields one file per partition value; use `coalesce(n)` for the opposite (fewer files, no shuffle) when the data is already small.

Common mistake: letting the default shuffle partitioning decide file count. A job with 200 shuffle partitions writes up to 200 files per output partition, so a small daily slice becomes hundreds of KB-sized files. Repartition or coalesce to a target file size before writing, and remember that `row_group_size` (rows per row group) and `maxRecordsPerFile` (rows per file) are different knobs.

## How do columnar storage and predicate pushdown make queries faster?

Parquet stores each column contiguously, so a scan is proportional to the columns selected, not the rows. On top of that, Parquet metadata enables three levels of skipping:

1. **Partition pruning** - the engine skips whole directories from the partition path (`ds=2026-10-03/`).
2. **Row-group skipping** - the footer stores per-column min/max and null counts, so a row group is skipped when the filter cannot match.
3. **Page/dictionary filtering** - finer-grained statistics and bloom filters can skip pages inside a row group.

The engine reads statistics in three stages. First it lists partition directories and drops those that cannot match. Then it opens each file footer and compares the predicate against the per-column-chunk min/max and null counts, skipping whole row groups. Then, if a page index is present (`write_page_index=True` in pyarrow, or a Parquet v2 writer), it can skip individual pages. Bloom filters add equality skipping on high-cardinality columns where broad min/max ranges are useless.

The `explain()` output is the interview evidence: look for `PushedFilters` and `PartitionFilters` rather than a full scan.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
curated = spark.read.parquet("s3://acme-curated/orders/")

filtered = curated.filter("order_date = '2026-10-04' AND country = 'DE'")
filtered.explain("formatted")
# Look for:
#   PartitionFilters: [isnotnull(order_date#..), (order_date#.. = 2026-10-04)]
#   PushedFilters:    [IsNotNull(country), EqualTo(country,DE)]

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: read the statistics an engine would use to skip row groups
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("orders/ds=2026-10-03/part-00000.parquet")
metadata = parquet_file.metadata

for rg in range(metadata.num_row_groups):
    row_group = metadata.row_group(rg)
    print(f"row group {rg}: {row_group.num_rows:,} rows, {row_group.total_byte_size:,} bytes")
    column = row_group.column(0)
    stats = column.statistics
    if stats is not None:
        print(f"  {column.path_in_schema}: min={stats.min} max={stats.max} nulls={stats.null_count}")

The same idea in a SQL engine (Athena over S3, or Trino):

```sql
SELECT country, SUM(amount) AS revenue
FROM parquet_scan('s3://acme-curated/orders/*/*.parquet')
WHERE order_date = DATE '2026-10-04'
  AND country = 'DE'
GROUP BY country;
```

Two design levers make skipping effective: **partition on the columns you filter most**, and **sort or cluster within files on the next most selective column** so that min/max ranges are tight. If a file mixes all dates, no row group can be skipped.

Common mistake: writing a filter that cannot be pushed down. A cast such as `WHERE CAST(order_ts AS DATE) = ...`, a function on the column, or a `LIKE '%x'` hides the raw column from the reader, so statistics cannot prune anything. Filter on the raw column and let the engine handle formatting. Also confirm that `write_statistics` was not turned off, because a file written without statistics can never be skipped.

## What are row groups, file sizes, and the small files problem?

A Parquet file is split into row groups (commonly 128 MB by default in Spark), and each row group contains one column chunk per column. Size choices trade off:

+ Larger row groups: better compression and fewer reads, but coarser skipping and more memory per read.
+ Smaller row groups: finer skipping, more overhead and worse compression.

Row-group size sets two different things. It is the unit of statistics-based skipping, so smaller row groups prune better. It is also the unit of parallelism inside one file: Spark can split a file only at row-group boundaries, so a 1 GB file with one 1 GB row group is read by a single task. The usual target is 64-128 MB per row group, which keeps pruning useful while still splitting a large file across tasks.

The bigger production problem is file count. Thousands of tiny files per partition hurt every engine: each file means one open, one footer parse, and one task.

Small-file math: a daily partition of 100 GB written as 1 MB files is about 100,000 files. Each file costs one object-store open, one footer parse, and one task, so the engine spends more time on metadata than on data. Compacted to 128 MB files the same partition is about 800 files. A common rule is to keep files between 128 MB and 1 GB and to keep the total file count per table in the tens of thousands, not millions.

```text
bad layout:  s3://acme-curated/orders/ds=2026-10-04/part-00000 ... part-04800  (KB-sized files)
good layout: s3://acme-curated/orders/ds=2026-10-04/part-00000 ... part-00007  (~128-512 MB files)
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Compact small files into fewer, larger ones (read, coalesce, rewrite)
small = spark.read.parquet("s3://acme-curated/orders/ds=2026-10-04/")
target_files = 8
(
    small.coalesce(target_files)
    .write.mode("overwrite")
    .parquet("s3://acme-curated/orders/ds=2026-10-04-compacted/")
)

# Tune the write side so compaction is needed less often
spark.conf.set("spark.sql.files.maxRecordsPerFile", 5_000_000)
spark.conf.set("spark.sql.files.openCostInBytes", 128 * 1024 * 1024)

On table formats, do not hand-compact: Iceberg and Delta expose maintenance (`rewrite_data_files`, `OPTIMIZE`) plus snapshot expiry, which is safer than rewriting files underneath a live table.

```bash
# Find the small-file problem before it becomes an incident
aws s3 ls s3://acme-curated/orders/ --recursive --summarize | tail -5
aws s3api list-objects-v2 --bucket acme-curated --prefix orders/ds=2026-10-04/ \
  --query 'length(Contents)' --output text

# How many files does one partition actually hold on HDFS?
hdfs dfs -count -v hdfs:///warehouse/orders/ds=2026-10-04/
```

Common mistake: compacting with `coalesce` and calling it done. `coalesce` reduces files without a shuffle, but it can produce uneven file sizes and it rewrites the whole partition. If the job runs daily on live data, use the table format's maintenance (`OPTIMIZE`, `rewrite_data_files`, `compact`) so commits stay atomic and readers keep a consistent snapshot.

## How does partitioning interact with Parquet?

The partition key becomes a directory level (`orders/ds=2026-10-04/country=DE/part-....parquet`), and the value is also stored in the file for safety. Partitioning is the highest-leverage performance decision and the easiest one to get wrong.

Partitioning is directory-based. The writer removes the partition columns from the file body by default and encodes the values in the path as `key=value` segments. The reader lists directories, parses those values, and injects them as columns. That listing is cheap on a few thousand directories and expensive on hundreds of thousands, which is the main reason over-partitioning hurts.

Guidelines:

+ Partition on low-to-medium cardinality columns used in filters: date, region, tenant, status.
+ Keep partition count manageable. A few thousand partitions per table is fine; hundreds of thousands is not.
+ Do not partition on high-cardinality keys such as `user_id` or millisecond timestamps - each partition becomes a tiny file.
+ Prefer one date grain: `ds=YYYY-MM-DD` for daily, `hour=` only when volume truly needs it.
+ For low-cardinality columns (like `country`), consider clustering or sorting inside files instead of an extra directory level.

Worked example: three years of daily orders is about 1,095 partitions. If each day is 1 GB, a 256 MB target gives about 4 files per day, so about 4,380 files for the table. Add `country` with 40 values and the same table becomes about 175,000 directories, most holding a few megabytes. That is the over-partitioning cliff: more directories, smaller files, and a larger metadata bill.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Over-partitioned: 120k directories, tiny files, slow metadata operations
orders.write.partitionBy("order_date", "hour", "customer_id").parquet("s3://acme-bad/orders/")

# Right-sized: day grain, country clustered inside files
(
    orders.repartition("order_date")
    .sortWithinPartitions("country", "order_id")
    .write.partitionBy("order_date")
    .parquet("s3://acme-good/orders/")
)

Hidden partitioning in Iceberg removes most of this manual reasoning: you declare `days(ts)` or `bucket(16, user_id)` and the engine handles pruning without exposing directory columns in queries.

Common mistake: partitioning on a high-cardinality column. `customer_id` or millisecond timestamps produce one directory per value, so files shrink below a useful size and every query pays a large listing cost. Use a few low-to-medium cardinality columns for directories and push higher-cardinality keys into sort or cluster order inside the files.

## What compression and encoding should you use?

Parquet compresses per column chunk and chooses encodings from the data type (dictionary encoding for repeated strings, run-length encoding for sorted values, delta encodings for timestamps). Codec choice is the knob you control:

Encoding and compression are separate stages. The writer first encodes values (dictionary, RLE, delta, bit-packing), then compresses the encoded bytes with the codec. Dictionary encoding only wins while the distinct values fit in the dictionary page (`dictionary_pagesize_limit`, 1 MiB by default); past that the column falls back to plain encoding and re-encoding costs CPU. A column of unique UUIDs compresses poorly; a column of 40 country codes compresses extremely well.

| Codec | Ratio | Speed | Splittable | Typical use |
|---|---|---|---|---|
| Snappy | moderate | fast | yes | Spark default; balanced hot path |
| Zstd | high | medium | yes | cold storage, cost-sensitive lakes |
| Gzip | high | slow | yes | legacy; splits at row groups, but decompression is CPU-heavy |
| LZ4 | low-moderate | fastest | yes | latency-critical reads |
| None | 1x | fastest CPU | yes | already compressed payloads |

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: the same data written with different codecs
import os
import pyarrow.parquet as pq

for codec in ["none", "snappy", "zstd", "gzip"]:
    path = f"orders/codec={codec}.parquet"
    pq.write_table(
        table,
        path,
        compression=codec,
        use_dictionary=True,
        write_statistics=True,
    )
    print(codec, os.path.getsize(path), "bytes")

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Per-write codec choice
orders.write.option("compression", "zstd").parquet("s3://acme-cold/orders/")

# Change it globally for a session
spark.conf.set("spark.sql.parquet.compression.codec", "zstd")

Also reduce the data itself: cast wide strings, drop unused columns before writing, and avoid storing raw JSON blobs as string columns. Encoding cannot fix a schema that stores 2 KB of unused JSON per row.

Common mistake: choosing a codec by ratio alone. Zstd at a high level costs CPU on every read of a hot table, and Gzip is slower still. Snappy or LZ4 usually wins for interactive queries, while Zstd wins for cold data and storage-billed lakes. Measure on your own data, because the ratio depends on cardinality, type, and sort order, not just on the codec name.

## How do nested types work in Parquet?

Parquet natively represents `struct`, `array`, and `map`, and Spark maps DataFrames onto them directly. Two rules keep nested writes efficient:

Nested values are flattened into leaf columns using repetition and definition levels, the model Parquet inherited from Dremel. A `struct` becomes one column chunk per leaf field; a `list<struct>` becomes a repeated group whose children are separate chunks. The levels record how many list elements and how many optional fields are present, so the reader can rebuild rows without storing offsets in the leaf data. Projecting `customer.country` therefore reads only that leaf chunk.

+ Read only the nested leaves you need with dot notation (`customer.country`) so the engine can skip the other sub-columns.
+ Do not `explode` large arrays earlier than necessary; exploding multiplies rows and destroys the columnar advantage.

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: nested struct and list become separate leaf column chunks
import pyarrow as pa
import pyarrow.parquet as pq

nested_schema = pa.schema([
    ("event_id", pa.string()),
    ("customer", pa.struct([
        ("customer_id", pa.int64()),
        ("country", pa.string()),
        ("tier", pa.string()),
    ])),
    ("items", pa.list_(pa.struct([
        ("sku", pa.string()),
        ("qty", pa.int32()),
        ("price", pa.decimal128(10, 2)),
    ]))),
])

table = pa.Table.from_pylist(nested_rows, schema=nested_schema)
pq.write_table(table, "events/part-00000.parquet", compression="zstd")

parquet_file = pq.ParquetFile("events/part-00000.parquet")
row_group = parquet_file.metadata.row_group(0)
for i in range(row_group.num_columns):
    print(row_group.column(i).path_in_schema)
# event_id
# customer.customer_id
# customer.country
# customer.tier
# items.list.element.sku
# items.list.element.qty
# items.list.element.price

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

events = spark.read.json("s3://acme-raw/events/")
events.printSchema()
# root
#  |-- event_id: string
#  |-- customer: struct (customer_id, country, tier)
#  |-- items: array<struct (sku, qty, price)>

# Project only the nested leaves required for the report
events.select(
    "event_id",
    "customer.country",
    F.explode("items").alias("item"),
).select("event_id", "country", "item.sku", "item.qty").show(5)

# Write the nested shape without flattening it
events.write.mode("overwrite").parquet("s3://acme-curated/events/")

Common mistake: exploding arrays to flatten them. `explode` multiplies rows, breaks the columnar advantage, and makes statistics on the exploded keys useless because one parent row now repeats many times. Keep data nested and let the engine project the leaves it needs; flatten only at the final reporting layer.

## How does schema evolution work with Parquet?

Parquet is schema-on-read: the schema lives in the file footer, and readers reconcile files with different schemas. Adding a nullable column at the end is safe. Removing or renaming columns, or changing a type incompatibly, needs care.

Schema evolution works because the schema is stored per file, not per dataset. The reader reads each file's footer, aligns columns by name, and fills missing columns with null. Adding a nullable column is safe: old files simply have no chunk for it. Widening an integer type is safe if the reader can upcast. Removing, renaming, or incompatibly retyping a column is not evolution, because old and new files disagree about meaning, not just about presence.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Batch read: merge schemas across files in the path
events = spark.read.option("mergeSchema", "true").parquet("s3://acme-curated/events/")

# Session-wide default, convenient for near-real-time readers
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: read two files with different schemas and unify them
import pyarrow as pa
import pyarrow.parquet as pq

old_file = pq.ParquetFile("events/ds=2026-10-01/part-00000.parquet")
new_file = pq.ParquetFile("events/ds=2026-10-04/part-00000.parquet")

merged = pa.unify_schemas([old_file.schema_arrow, new_file.schema_arrow])
print("merged schema:", merged)

# Pin the merged schema on the read so every file is reconciled the same way
table = pq.read_table("events/", schema=merged)

Rules that avoid incidents:

+ Add columns as nullable; never reuse a column name with a new meaning.
+ `mergeSchema` on every read slows planning for wide schemas; prefer catalog-driven tables (Iceberg/Delta/Hive metastore) over schema inference in production.
+ Renames are migrations, not evolutions. Write the new name and backfill, or use a view that maps old to new.
+ Parquet has no schema registry. If producers change types freely, add Avro + a registry in the ingestion path, or enforce a contract in the table format.

Common mistake: relying on `mergeSchema` forever. Reading with schema inference or schema merging forces the reader to open every footer before planning, which is slow on a table with millions of files and can produce surprising types. Once a table is stable, pin an explicit schema or register it in a catalog; merge only at ingestion boundaries.

## What are common Parquet mistakes in production?

+ **Writing thousands of small files.** Use fewer, larger files and schedule compaction.
+ **Over-partitioning.** Every extra partition column multiplies directories and shrinks files.
+ **Partitioning by a column you never filter on.** You pay metadata cost and get no pruning.
+ **Non-splittable codecs.** A single gzip CSV cannot be split; gzip Parquet reads can still bottleneck.
+ **No sort/clustering inside files.** Wide min/max ranges make row-group skipping useless.
+ **Blind `append`.** Retries and backfills duplicate rows; make loads idempotent or use a merge.
+ **Storing everything as string.** You lose statistics quality, encoding, and type safety.
+ **Using Parquet for upserts.** Use Iceberg, Delta, or Hudi when records change.
+ **Forgetting timezone semantics.** Store timestamps consistently (UTC) and document it; naive timestamps across engines cause off-by-hours bugs.
+ **Reading with schema inference in production.** It is slow and can silently change types; pass an explicit schema.

Most Parquet incidents reduce to four numbers: file count, file size, row-group size, and sort order. Check the number of files per partition, the median file size, the number of row groups per file, and whether the column you filter on is sorted inside the file. A healthy table has hundreds of files of 128 MB to 1 GB, a handful of row groups each, and tight min/max ranges on the filter columns.

Two less obvious traps. First, `write_statistics=False` or a writer that omits statistics makes row-group skipping impossible, silently. Second, mixing time zones or storing naive timestamps makes min/max comparisons wrong across engines, so a filter that should prune does not, or prunes the wrong rows.

## How do you inspect Parquet metadata and file layout?

Being able to inspect a file is what separates a working pipeline from a debuggable one.

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: row groups, sizes, statistics, compression
import pyarrow.parquet as pq

pf = pq.ParquetFile("orders/ds=2026-10-04/part-00000.snappy.parquet")
print(pf.metadata.num_rows, pf.metadata.num_row_groups)
print(pf.schema_arrow)

rg = pf.metadata.row_group(0)
for i in range(rg.num_columns):
    col = rg.column(i)
    stats = col.statistics
    print(col.path_in_schema, col.compression, stats.min if stats else None, stats.max if stats else None)

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: the full metadata surface for one file
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("orders/ds=2026-10-04/part-00000.snappy.parquet")
metadata = parquet_file.metadata

print("created by:", metadata.created_by)
print("format version:", metadata.format_version)
print("rows:", metadata.num_rows, "row groups:", metadata.num_row_groups)
print("footer bytes:", metadata.serialized_size)

for rg in range(metadata.num_row_groups):
    row_group = metadata.row_group(rg)
    print(f"row group {rg}: {row_group.num_rows} rows, "
          f"{row_group.total_byte_size} bytes, columns={row_group.num_columns}")
    for c in range(row_group.num_columns):
        column = row_group.column(c)
        stats = column.statistics
        print(f"  {column.path_in_schema}: {column.compression} {column.encodings} "
              f"min={getattr(stats, 'min', None)} max={getattr(stats, 'max', None)}")

```bash
# File counts and sizes per partition - the first check for small-file problems
aws s3 ls s3://acme-curated/orders/ds=2026-10-04/ --recursive --human-readable --summarize

# One-line schema and row-group summary
python -c "import pyarrow.parquet as pq; print(pq.ParquetFile('part-00000.snappy.parquet').metadata)"
```

Common mistake: assuming statistics are always present. Small row groups, all-null columns, and writers configured with `write_statistics=False` produce `None` statistics, so code that reads `stats.min` without a guard crashes or reports nonsense. Also note that `pyarrow.parquet.ParquetDataset` is deprecated in favor of `pyarrow.dataset.dataset`, which supports the same partitioned layout plus filtering and projection.

## Parquet vs ORC vs Avro: when do you choose which?

| Format | Orientation | Strengths | Best for | Watch out for |
|---|---|---|---|---|
| Parquet | columnar | ecosystem, nested types, predicate pushdown | lakehouse and Spark/Trino/Athena analytics | no row-level updates on its own |
| ORC | columnar | Hive integration, ACID in Hive, strong compression | Hive-centric warehouses | smaller ecosystem outside Hive |
| Avro | row | compact, schema registry, fast row writes | Kafka topics, landing/streaming, record-at-a-time | poor for analytical scans |
| JSON/CSV | row text | debuggable, universal | raw landing, small inputs | size, parsing cost, no types |

The formats differ in the unit that gets compressed and in how rows are reassembled. Parquet and ORC are both columnar with row-group-like stripes, but Parquet has the broader ecosystem and stronger nested-type support across Spark, Trino, DuckDB, BigQuery, and Athena; ORC is strongest inside Hive and has built-in Hive ACID support. Avro is row-oriented with compact binary encoding and a schema registry story, which is why it is common on Kafka topics. None of them provides transactions on its own: Delta, Iceberg, and Hudi add a metadata log, ACID commits, snapshots, and time travel on top of Parquet files.

A common lakehouse pattern uses both: Avro or JSON for the streaming/landing layer, Parquet for the curated analytical layer, with Iceberg or Delta providing transactions and time travel.

Common mistake: asking which format is best without naming the access pattern. Choose by how the data is read: analytical scans favor Parquet or ORC, row-at-a-time streaming favors Avro, and mutable tables favor a table format over raw Parquet. A lakehouse commonly uses Avro or JSON at the landing edge and Parquet under Iceberg or Delta for the curated layer.

## How do you tune Parquet files for performance?

Parquet tuning is four decisions in order: how many files, how big each file is, how the data is ordered inside each file, and how it is encoded and compressed. Get the first three right and the format does the work; tune codecs only after the layout is correct. Every engine exposes the same knobs under different names, so the reasoning transfers from Spark to Trino, DuckDB, Athena, and Snowflake unloads.

Target 128 MB to 1 GB per file. Below about 128 MB, per-file overhead (open, footer parse, task scheduling) dominates. Above about 1 GB, a single file becomes hard to rewrite and can limit parallelism if it has too few row groups. Pick a target, then compute file count as `total_size / target_size` and repartition to that count.

Worked example: 2 TB of curated orders with a 512 MB target is about 4,096 files. At 128 MB it is 16,384 files; at 1 MB it would be two million files and unusable. Set `spark.sql.files.maxPartitionBytes` and use `repartition(n)` or `coalesce(n)` so each output partition is roughly the target size.

Aim for row groups of 64-128 MB, which is usually a few hundred thousand to a couple of million rows depending on width. Row groups are the unit of statistics skipping and of intra-file parallelism, so too few row groups means one task reads a whole file, and too many means per-row-group overhead and weaker compression. In pyarrow this is `row_group_size` in rows; in Spark it is `spark.sql.parquet.rowGroupSize` in bytes.

In [ ]:
# ILLUSTRATIVE - requires pyarrow: install to run, or read on.
# pyarrow: a tuned write - row groups sized for pruning, pages sized for the codec
import pyarrow.parquet as pq

pq.write_table(
    table,
    "orders/ds=2026-10-04/part-00000.zstd.parquet",
    row_group_size=2_000_000,        # rows per row group
    data_page_size=1 * 1024 * 1024,  # 1 MiB target page before compression
    compression="zstd",
    compression_level=3,             # 1-3 balances ratio and CPU
    use_dictionary=True,
    write_statistics=True,           # never disable; skipping depends on this
    write_page_index=True,           # finer page-level skipping
)

```sql
-- Spark: size the write for the target file count
SET spark.sql.files.maxPartitionBytes = 536870912;   -- 512 MB per task
SET spark.sql.parquet.compression.codec = zstd;
SET spark.sql.parquet.rowGroupSize = 134217728;      -- 128 MB row groups

-- Delta Lake: compact small files and order rows inside files
OPTIMIZE analytics.orders ZORDER BY (country, order_ts);

-- Iceberg: rewrite data files and manifests, then drop old snapshots
CALL catalog.system.rewrite_data_files(table => 'analytics.orders');
CALL catalog.system.rewrite_manifests('analytics.orders');
CALL catalog.system.expire_snapshots('analytics.orders', TIMESTAMP '2026-10-01 00:00:00');
```

Compression trades storage and I/O against CPU. Snappy and LZ4 keep CPU low for hot, interactive reads; Zstd level 1-3 gives a better ratio at moderate cost; Gzip and high Zstd levels belong on cold data written once and read rarely. Before tuning the codec, sort or cluster the data on the most selective filter column: tight min/max ranges let the reader skip whole row groups, which saves far more I/O than a better codec.

A short checklist:
+ Write statistics (`write_statistics=True`) and always read with column projection.
+ Sort or bucket within files on the column you filter most, so min/max ranges are tight.
+ Keep row groups per file in the tens, not the thousands.
+ Compact small files on a schedule, and use the table format's maintenance instead of hand-rewriting live partitions.
+ Re-measure after each change: bytes scanned, files read, and plan time are the numbers that move.

Common mistake: tuning the codec first. A file with one huge row group and unsorted values cannot be pruned no matter how well it compresses, so every query decompresses everything. Fix layout before compression.

## Real world use cases for Parquet

### 1. Event lake on S3 queried by Athena and Spark

Clickstream events land as JSON, a batch job converts them to partitioned Parquet, and analysts query with Athena. Daily partitions (`ds=YYYY-MM-DD`) allow partition pruning; hourly partitions are added only for the two highest-volume flows.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
(
    spark.read.json("s3://acme-raw/clicks/ds=2026-10-04/")
    .select("event_id", "user_id", "url", "event_ts", "device.country")
    .withColumn("ds", F.lit("2026-10-04"))
    .repartition(16)
    .write.mode("overwrite")
    .option("compression", "zstd")
    .parquet("s3://acme-curated/clicks/ds=2026-10-04/")
)

Why Parquet: Athena bills by bytes scanned, so columnar projection plus partition pruning is a direct cost lever.

### 2. Offloading a warehouse to object storage

A fintech exports large cold fact tables from Snowflake to Parquet monthly, keeping only recent months in the warehouse. Historical analysis runs on Spark or Trino against S3 at a fraction of the cost.

```sql
-- Snowflake: unload one month as partitioned Parquet
COPY INTO @acme_stage/orders/month=2026-09/
FROM analytics.orders
WHERE order_date >= '2026-09-01' AND order_date < '2026-10-01'
FILE_FORMAT = (TYPE = PARQUET COMPRESSION = ZSTD)
PARTITION_BY = (order_date)
HEADER = TRUE;
```

Why Parquet: it is portable across engines, so the cold copy is not locked into one vendor.

### 3. Training snapshots for machine learning

A feature platform materializes a daily training snapshot as Parquet so training jobs read a stable, immutable dataset with the same schema for every experiment.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
features = spark.read.parquet("s3://acme-features/training/ds=2026-10-04/")
features.select("user_id", "label", "days_since_signup", "orders_30d").write.mode("overwrite").parquet(
    "s3://acme-ml/snapshots/ds=2026-10-04/"
)

Why Parquet: immutable snapshots make experiments reproducible, and column projection keeps training reads fast even with hundreds of features.

### 4. CDC landing zone feeding a lakehouse table

Debezium writes change events to Kafka; a sink writes micro-batch Parquet into a landing prefix; a merge job applies changes to an Iceberg table. Parquet is the transport format, Iceberg is the queryable table.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
landing = spark.read.parquet("s3://acme-landing/customers/dt=2026-10-04/")
landing.createOrReplaceTempView("stg_customers")

spark.sql("""
    MERGE INTO iceberg.customers AS t
    USING stg_customers AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

Why this split: Parquet is cheap to write and engine-neutral, while the table format provides ACID merges, snapshot isolation, and time travel.

### 5. Log analytics with selective columns

A SaaS platform stores 5 TB/day of application logs. Only a few columns are queried regularly, so the pipeline writes Parquet with a compact set of typed columns plus a compressed `payload` string, and separates high-volume debug logs into a Zstd-compressed cold prefix.

Why Parquet: projection and compression turn an unaffordable full-text scan into a targeted, cheap query, and the same files are readable by Spark, Trino, and DuckDB.

Across all five patterns the same levers repeat: write typed, narrow columns; partition on the columns you filter on most; keep files between 128 MB and 1 GB; sort or cluster inside files so statistics can skip row groups; compress cold data more aggressively than hot data; and put mutable tables behind Iceberg or Delta instead of raw Parquet. The five cases differ in access pattern, not in the format mechanics.

## Parquet exercises

These exercises use only the Python standard library and model Parquet layout arithmetic with small, runnable calculations. Each one names the task, gives a hint, states the expected result, then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are estimates, not promises. Real file sizes depend on types, cardinality, sort order, and codec; the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Estimate the file count for a dataset

**Task.** For a 2 TiB dataset, compute how many files you get at target sizes of 128 MB, 512 MB, and 1 GB.

**Hint.** `files = ceil(total_bytes / target_bytes)`; use `math.ceil`.

**Expected result.** 16,384 files at 128 MB, 4,096 at 512 MB, and 2,048 at 1 GB.

In [ ]:
import math

TOTAL_BYTES = 2 * 1024**4   # 2 TiB
counts = {mb: math.ceil(TOTAL_BYTES / (mb * 1024**2)) for mb in [128, 512, 1024]}
for mb, n in counts.items():
    print(f"target {mb:>4} MB -> {n:>6,} files")

assert counts[128] == 16_384
assert counts[512] == 4_096
assert counts[1024] == 2_048

### Try it 2. Compression ratio and storage math

**Task.** 200 GB of JSON becomes 45 GB as Snappy Parquet and 35 GB as Zstd Parquet. Compute the compression ratio of each and the monthly storage cost of all three at 0.02 per GB-month.

**Hint.** `ratio = original / stored` and `cost = stored_gb * price`.

**Expected result.** Snappy is 4.44x and Zstd is 5.71x; Zstd costs 0.70 per month against 4.00 for raw JSON, saving 3.30.

In [ ]:
original_gb = 200
snappy_gb = 45
zstd_gb = 35
price_per_gb_month = 0.02

for name, gb in [("json", original_gb), ("snappy", snappy_gb), ("zstd", zstd_gb)]:
    ratio = original_gb / gb
    cost = gb * price_per_gb_month
    print(f"{name:7} {gb:>4} GB  ratio={ratio:5.2f}x  cost=${cost:5.2f}/month")

savings = original_gb * price_per_gb_month - zstd_gb * price_per_gb_month
print(f"zstd saves ${savings:.2f} per month vs raw JSON")

assert round(original_gb / snappy_gb, 2) == 4.44
assert round(original_gb / zstd_gb, 2) == 5.71
assert round(savings, 2) == 3.30

### Try it 3. Row-group count versus threads

**Task.** A 4 GB file is read by 16 threads. Compute the number of row groups for row-group sizes of 64 MB, 128 MB, and 512 MB, and say whether the file is split finely enough to keep every thread busy.

**Hint.** `row_groups = ceil(file_mb / row_group_mb)`; enough means at least one row group per thread.

**Expected result.** 64 and 128 MB give 64 and 32 row groups (enough); 512 MB gives only 8 row groups, so half the threads idle.

In [ ]:
import math

FILE_MB = 4096
THREADS = 16

for rg_mb in [64, 128, 512]:
    groups = math.ceil(FILE_MB / rg_mb)
    per_thread = groups / THREADS
    verdict = "enough" if groups >= THREADS else "too coarse"
    print(f"row_group={rg_mb:>4} MB -> {groups:>3} row groups, "
          f"{per_thread:.2f} per thread, {verdict}")

assert math.ceil(FILE_MB / 64) == 64
assert math.ceil(FILE_MB / 128) == 32
assert math.ceil(FILE_MB / 512) == 8

### Try it 4. Dictionary-encoding break-even

**Task.** A column has 1,000,000 rows, 24 bytes per value, and a 4-byte dictionary index. Model bytes as `distinct * value_bytes + rows * index_bytes` for dictionary encoding and `rows * value_bytes` for plain. Compute both at 1,000 and at 1,000,000 distinct values, and find the break-even distinct count.

**Hint.** Dictionary encoding wins while `distinct * (value_bytes - index_bytes) < rows * value_bytes`; solve for `distinct`.

**Expected result.** Dictionary wins at 1,000 distinct values and loses at 1,000,000; break-even is about 833,333 distinct values.

In [ ]:
VALUE_BYTES = 24
INDEX_BYTES = 4
ROWS = 1_000_000


def dictionary_bytes(distinct):
    return distinct * VALUE_BYTES + ROWS * INDEX_BYTES


def plain_bytes():
    return ROWS * VALUE_BYTES


print(f"plain = {plain_bytes():,} bytes")
for distinct in [1_000, 100_000, 1_000_000]:
    d = dictionary_bytes(distinct)
    print(f"distinct={distinct:>9,} dict={d:>12,} wins={d < plain_bytes()}")

break_even = (VALUE_BYTES - INDEX_BYTES) * ROWS / VALUE_BYTES
print(f"break-even distinct values ~= {break_even:,.0f}")

assert dictionary_bytes(1_000) < plain_bytes()
assert dictionary_bytes(1_000_000) > plain_bytes()
assert round(break_even) == 833_333

### Try it 5. Cost of reading one column versus all

**Task.** A file has 40 equal-sized columns and is 800 MB in total. Compute the bytes read when a query selects 3 columns, 5 columns, and all 40, and the percentage of the file each represents.

**Hint.** `bytes_read = selected_columns * (total_mb / column_count)`.

**Expected result.** 3 columns read 60 MB (7.5%), 5 columns read 100 MB (12.5%), and all 40 read 800 MB.

In [ ]:
TOTAL_MB = 800
COLUMNS = 40
COLUMN_MB = TOTAL_MB / COLUMNS


def scan(selected):
    return selected * COLUMN_MB


for n in [3, 5, 40]:
    mb = scan(n)
    print(f"{n:>2} columns -> {mb:6.1f} MB ({mb / TOTAL_MB * 100:5.1f}% of the file)")

assert round(scan(3), 1) == 60.0
assert round(scan(3) / TOTAL_MB * 100, 1) == 7.5
assert round(scan(40), 1) == 800.0

### Try it 6. Small-file compaction savings

**Task.** Each file costs 5 ms of open and footer-parse time. Compare 100,000 files of 1 MB with the same data compacted into 800 files of about 125 MB. Compute the metadata time before and after and the saving.

**Hint.** `seconds = files * overhead_ms / 1000`.

**Expected result.** 500 s before, 4 s after, a saving of 496 s.

In [ ]:
OVERHEAD_MS = 5
before_files = 100_000
after_files = 800

before = before_files * OVERHEAD_MS / 1000
after = after_files * OVERHEAD_MS / 1000
saved = before - after
print(f"metadata time: before {before:,.0f}s, after {after:.0f}s, saved {saved:,.0f}s")

assert before == 500.0
assert after == 4.0
assert saved == 496.0

### Try it 7. Choose a codec by CPU and storage tradeoff

**Task.** For 1,000 GB of raw data scanned 10 times a month, compute the monthly cost of four codecs. Storage costs 0.023 per GB-month and CPU costs 0.001 per GB scanned for each unit of relative CPU. Total cost is `stored_gb * storage_price + stored_gb * scans * cpu_factor * cpu_price`. Pick the cheapest.

**Hint.** `stored_gb = raw_gb * ratio`, then `min(totals, key=totals.get)`.

**Expected result.** Zstd is cheapest at about 7.38 per month; Gzip loses to CPU at 9.01, Snappy is 8.25, and no compression is 25.00 on storage.

In [ ]:
STORAGE_PRICE = 0.023      # per GB-month
SCANS = 10                 # full scans per month
CPU_PRICE = 0.001          # per GB-scan of relative CPU
RAW_GB = 1000

# codec -> (compression ratio, relative CPU per byte scanned)
codecs = {
    "none":   (1.00, 0.2),
    "snappy": (0.25, 1.0),
    "zstd":   (0.18, 1.8),
    "gzip":   (0.17, 3.0),
}

totals = {}
for name, (ratio, cpu) in codecs.items():
    stored = RAW_GB * ratio
    total = stored * STORAGE_PRICE + stored * SCANS * cpu * CPU_PRICE
    totals[name] = total
    print(f"{name:7} stored={stored:6.0f} GB  monthly cost=${total:6.2f}")

cheapest = min(totals, key=totals.get)
print("cheapest:", cheapest)

assert cheapest == "zstd"
assert round(totals["zstd"], 2) == 7.38
assert round(totals["gzip"], 2) == 9.01
assert round(totals["none"], 2) == 25.00

### Try it 8. Row-group skipping with min/max statistics

**Task.** A sorted file has 200 row groups of 1,000 rows each. Row group `i` holds values from `i * 1000` to `i * 1000 + 999`. A filter matches values from 80,000 to 119,999. Count how many row groups are read and how many are skipped by min/max statistics.

**Hint.** Skip a row group when `group_max < low` or `group_min > high`.

**Expected result.** 40 row groups are read and 160 are skipped, an 80% skip rate.

In [ ]:
GROUPS = 200
GROUP_SIZE = 1000
LOW = 80 * GROUP_SIZE
HIGH = 119 * GROUP_SIZE + (GROUP_SIZE - 1)

read = 0
skipped = 0
for i in range(GROUPS):
    group_min = i * GROUP_SIZE
    group_max = group_min + GROUP_SIZE - 1
    if group_max < LOW or group_min > HIGH:
        skipped += 1
    else:
        read += 1

print(f"read {read} of {GROUPS} row groups, skipped {skipped} "
      f"({skipped / GROUPS * 100:.0f}%)")

assert read == 40
assert skipped == 160
assert skipped / GROUPS == 0.8